# 智能文本生成系统性能测试 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

In [ ]:
import asyncio
import random
import time

class TextGeneratorSimulator:
    def __init__(self, delay_ms=200, error_rate=0.02):
        self.delay_ms = delay_ms
        self.error_rate = error_rate

    async def generate(self, prompt):
        await asyncio.sleep(self.delay_ms / 1000)
        if random.random() < self.error_rate:
            raise RuntimeError("模拟请求失败")
        return f"{prompt}，这是模拟续写。"

simulator = TextGeneratorSimulator()
prompts = ["今天天气", "智能电表", "设备巡检", "故障预警", "客户服务",
           "数据质量", "模型训练", "安全生产", "节能方案", "应急处置"]

async def send_request(number, prompt):
    started = time.perf_counter()
    try:
        answer = await simulator.generate(prompt)
        return {"number": number, "ok": True,
                "seconds": time.perf_counter() - started, "answer": answer}
    except Exception as error:
        return {"number": number, "ok": False,
                "seconds": time.perf_counter() - started, "error": str(error)}

def report(label, results, elapsed):
    successful = [item for item in results if item["ok"]]
    times = [item["seconds"] for item in results]
    print(label, "成功率", round(len(successful) / len(results), 2),
          "平均响应时间", round(sum(times) / len(times), 3),
          "吞吐量", round(len(results) / elapsed, 2),
          "最短", round(min(times), 3), "最长", round(max(times), 3))

async def main():
    started = time.perf_counter()
    sequential = []
    for number, prompt in enumerate(prompts, 1):
        sequential.append(await send_request(number, prompt))
    report("顺序", sequential, time.perf_counter() - started)

    semaphore = asyncio.Semaphore(5)
    async def limited(number, prompt):
        async with semaphore:
            return await send_request(number, prompt)

    started = time.perf_counter()
    concurrent = await asyncio.gather(
        *(limited(number, prompt) for number, prompt in enumerate(prompts, 1))
    )
    report("5 人并发", concurrent, time.perf_counter() - started)

await main()
